# Compare Concept, Rationale, and Attribution Prompt-Type Scores

This notebook compares score distributions across explanation families. It reads a
`data/consim_{MODEL_FILE}.csv` score file, infers an explanation family for each
row from its prompt type / method, and summarizes prompt-type scores with the
same 10-seed bucket convention used elsewhere in this repo.

The current score CSV schema has no explicit `family` column, so the family
mapping is intentionally visible and configurable below. Baseline rows (`B1`,
`B2`, `AB1`, `AB2`) are kept as `family=baseline` and their specification is
ignored, because baseline prompts are intended to be identical across families.

## Parameters

In [ ]:
# Score CSV: data/consim_{MODEL_FILE}_v2.csv
MODEL_FILE = "Qwen_Qwen3.5-9B"

# Optional filters. None = keep everything present in the CSV.
DATASETS = None              # e.g. ["RT", "AG"]
CLASSES_SUBSETS = None       # e.g. ["[0, 1]", "[0, 1, 2, 3]"]
SPECIFICATIONS = ["new_consim", "attributions", "rationales"]        # e.g. ["new_consim"] or ["old_consim"]
FAMILIES = None              # e.g. ["concepts", "rationales", "attributions"]
PROMPT_TYPES = None          # e.g. ["B1", "B2", "C1", "R1", "A1"]
METHODS = None               # e.g. ["SemiNMF", "saliency"]

# Aggregate raw rows to one value per seed before forming 10-seed buckets.
SEEDS_PER_BUCKET = 10
EXPECTED_SEEDS = 50

FAMILY_ORDER = ["concepts", "rationales", "attributions"]
PROMPT_TYPE_ORDER = ["B1", "B2", "AB1", "AB2", "C1", "C2", "C3", "AC1", "AC2", "AC3", "R1", "A1"]

## Imports

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO_ROOT = Path(".").resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.registries import CONCEPT_METHOD_NAMES, ATTRIBUTION_METHOD_NAMES

CSV_PATH = REPO_ROOT / "data" / f"consim_{MODEL_FILE}_v2.csv"
assert CSV_PATH.exists(), f"Score CSV not found: {CSV_PATH}"

## Load, Infer Family, and Filter

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df)} rows from {CSV_PATH.name}")
print(f"  datasets:       {sorted(df['dataset'].unique())}")
print(f"  specifications: {sorted(df['specification'].unique())}")
print(f"  prompt_types:   {sorted(df['prompt_type'].unique())}")
print(f"  methods:        {sorted(df['method'].unique())[:20]}{' ...' if df['method'].nunique() > 20 else ''}")
df.head()

In [ ]:
CONCEPT_METHODS = set(CONCEPT_METHOD_NAMES.values()) | set(CONCEPT_METHOD_NAMES.keys())
ATTRIBUTION_METHODS = set(ATTRIBUTION_METHOD_NAMES)
# Add common display variants in case score rows store class names rather than CLI keys.
ATTRIBUTION_METHODS |= {
    "Saliency",
    "IntegratedGradients",
    "Integrated Gradients",
    "SmoothGrad",
    "SquareGrad",
    "VarGrad",
    "GradientShap",
    "LIME",
    "KernelShap",
    "Occlusion",
    "Sobol",
}
RATIONALE_METHODS = {"meta-llama/Llama-3.2-3B-Instruct", "Qwen/Qwen3.5-2B"}

BASELINE_PROMPT_TYPES = {"B1", "B2"}  #, "AB1", "AB2"}
CONCEPT_PROMPT_TYPES = {"C1", "C2", "C3"}  #, "AC1", "AC2", "AC3"}
RATIONALE_PROMPT_TYPES = {"R1"}
ATTRIBUTION_PROMPT_TYPES = {"A1"}

def infer_family(row: pd.Series) -> str:
    pt = str(row["prompt_type"])
    method = str(row["method"])
    if pt in CONCEPT_PROMPT_TYPES or method in CONCEPT_METHODS:
        return "concepts"
    if pt in RATIONALE_PROMPT_TYPES or method in RATIONALE_METHODS:
        return "rationales"
    if pt in ATTRIBUTION_PROMPT_TYPES or method in ATTRIBUTION_METHODS:
        return "attributions"
    if pt in BASELINE_PROMPT_TYPES or method == "baseline":
        return "baseline"
    return "unknown"

df = df.copy()
df["family_raw"] = df.apply(infer_family, axis=1)
df["family"] = df["family_raw"]

print(df.groupby(["family_raw", "prompt_type", "method"]).size().reset_index(name="n").sort_values(["family_raw", "prompt_type", "method"]).to_string(index=False))

In [ ]:
def _apply_filter(frame: pd.DataFrame, column: str, values) -> pd.DataFrame:
    if values is None:
        return frame
    values = list(values)
    missing = set(values) - set(frame[column].unique())
    if missing:
        print(f"  warning: {column} values not present before filtering: {sorted(missing)}")
    return frame[frame[column].isin(values)]

selected_families = FAMILY_ORDER if FAMILIES is None else list(FAMILIES)

df_f = df.copy()
df_f = _apply_filter(df_f, "dataset", DATASETS)
df_f = _apply_filter(df_f, "classes_subset", CLASSES_SUBSETS)
df_f = _apply_filter(df_f, "specification", SPECIFICATIONS)
df_f = _apply_filter(df_f, "prompt_type", PROMPT_TYPES)
df_f = _apply_filter(df_f, "method", METHODS)

# Keep selected explanation families plus baseline rows. Baselines are not
# family-specific: B1/B2/AB1/AB2 are byte-identical across families/methods.
# Do NOT replicate them into every family here, otherwise rows such as
# specification=attributions, family=concepts can be fabricated. Instead, keep
# them as family=baseline and normalize specification so the bucketing code
# averages equal baseline keys across specifications at the seed level.
non_baseline = df_f[df_f["family_raw"].isin(selected_families)].copy()
baseline = df_f[df_f["family_raw"] == "baseline"].copy()
baseline["family"] = "baseline"
baseline["specification"] = "baseline"
df_plot = pd.concat([non_baseline, baseline], ignore_index=True)

print(f"Rows after filters and baseline handling: {len(df_plot)}")
display(df_plot.groupby(["family", "family_raw", "prompt_type", "specification"]).size().reset_index(name="n"))

## 10-Seed Bucket Statistics

In [ ]:
def _bucket_stats(group: pd.DataFrame) -> pd.Series:
    # Average multiple rows per seed first (e.g. multiple concept methods) so
    # each seed has equal weight in the bucket mean.
    g = group.groupby("seed", as_index=False)["score"].mean().sort_values("seed")
    n_seeds = len(g)
    if n_seeds != EXPECTED_SEEDS:
        print(f"  warning: group={group.name!r} has {n_seeds} distinct seeds (expected {EXPECTED_SEEDS})")
    n_buckets = n_seeds // SEEDS_PER_BUCKET
    if n_buckets == 0:
        return pd.Series({"mean_score": np.nan, "std_score": np.nan, "n_buckets": 0, "n_seeds": n_seeds})
    vals = g["score"].to_numpy()[: n_buckets * SEEDS_PER_BUCKET]
    bucket_means = vals.reshape(n_buckets, SEEDS_PER_BUCKET).mean(axis=1)
    return pd.Series({
        "mean_score": float(bucket_means.mean()),
        "std_score": float(bucket_means.std(ddof=1)) if n_buckets > 1 else 0.0,
        "n_buckets": int(n_buckets),
        "n_seeds": int(n_seeds),
    })

GROUP_COLS = ["dataset", "classes_subset", "specification", "family", "prompt_type"]
stats = (
    df_plot.groupby(GROUP_COLS, dropna=False)
    .apply(_bucket_stats, include_groups=False)
    .reset_index()
)

# Stable display order.
stats["family"] = pd.Categorical(stats["family"], categories=FAMILY_ORDER + ["baseline"], ordered=True)
stats["prompt_type"] = pd.Categorical(stats["prompt_type"], categories=PROMPT_TYPE_ORDER, ordered=True)
stats = stats.sort_values(["dataset", "classes_subset", "specification", "family", "prompt_type"]).reset_index(drop=True)
stats

## Best Non-Baseline Method per Family

For each `family`, select one best non-baseline explanation configuration
overall across the currently filtered rows, using the same 10-seed bucket mean.

Configuration identity:
- Concepts: `(method, interpretation)`
- Rationales: LLM name stored in `method`
- Attributions: attribution `method`

Excluded from selection: baselines, `ClassesAs` / `classes`, and anonymized
prompt types (`AB*`, `AC*`, `AR*`, `AA*`). The selected best config for a
family is then reused in every `(dataset, classes_subset)` panel. The final
plot compares prompt types from those best family configs plus
non-anonymized baselines (`B1`, `B2`), ranked from highest to lowest score
within each panel. `specification` is ignored in these final panels: equal
keys are averaged per seed before 10-seed bucketing, which also makes the
baseline invariant explicit.

In [ ]:
NON_ANON_PROMPT_TYPES_BY_FAMILY = {
    "concepts": {"C1", "C2", "C3"},
    "rationales": {"R1"},
    "attributions": {"A1"},
}
NON_ANON_BASELINE_PROMPT_TYPES = {"B1", "B2"}
EXCLUDED_BEST_METHODS = {"baseline", "classes", "ClassesAs"}

def _family_config(row: pd.Series) -> str:
    family = row["family"]
    method = str(row["method"])
    interpretation = row.get("interpretation")
    if family == "concepts":
        if pd.isna(interpretation) or str(interpretation) in {"None", "nan", "<NA>"}:
            return method
        return f"{method} / {interpretation}"
    # For rationales the LLM is stored in `method`; for attributions this is
    # the attribution method.
    return method

# Candidate rows for selecting the best method/config per family.
best_candidates = df_plot.copy()
best_candidates = best_candidates[best_candidates["family_raw"] != "baseline"].copy()
best_candidates = best_candidates[~best_candidates["method"].astype(str).isin(EXCLUDED_BEST_METHODS)].copy()
best_candidates["family_config"] = best_candidates.apply(_family_config, axis=1)

allowed_non_anon_prompt_types = set().union(*NON_ANON_PROMPT_TYPES_BY_FAMILY.values())
best_candidates = best_candidates[best_candidates["prompt_type"].isin(allowed_non_anon_prompt_types)].copy()
best_candidates = best_candidates[
    best_candidates.apply(
        lambda r: r["prompt_type"] in NON_ANON_PROMPT_TYPES_BY_FAMILY.get(r["family"], set()),
        axis=1,
    )
]

# Select the best config globally per family over the current filters. This
# intentionally does NOT condition on dataset/classes_subset/specification.
BEST_CONFIG_GROUP_COLS = ["family", "family_config"]
if best_candidates.empty:
    best_config_stats = pd.DataFrame(columns=BEST_CONFIG_GROUP_COLS + ["mean_score", "std_score", "n_buckets", "n_seeds"])
    best_configs = pd.DataFrame(columns=BEST_CONFIG_GROUP_COLS + ["mean_score", "std_score", "n_buckets", "n_seeds"])
    print("No non-baseline, non-anonymized candidate rows found.")
else:
    best_config_stats = (
        best_candidates.groupby(BEST_CONFIG_GROUP_COLS, dropna=False)
        .apply(_bucket_stats, include_groups=False)
        .reset_index()
        .dropna(subset=["mean_score"])
    )
    best_configs = (
        best_config_stats.sort_values("mean_score", ascending=False)
        .groupby(["family"], as_index=False, sort=False)
        .head(1)
        .sort_values(["family"])
        .reset_index(drop=True)
    )

best_configs

## Ranked Bar Plots: Best Family Configs + Baselines

In [ ]:
# Keep only rows belonging to the selected best configs, then aggregate by
# non-anonymized prompt_type with the same seed-bucket convention.
if best_configs.empty:
    best_expl_rows = best_candidates.iloc[0:0].copy()
else:
    best_keys = best_configs[BEST_CONFIG_GROUP_COLS].drop_duplicates()
    best_expl_rows = best_candidates.merge(best_keys, on=BEST_CONFIG_GROUP_COLS, how="inner")

# Final comparison panels are per (dataset, classes_subset), not per
# specification. If the same seed/key appears under multiple specifications,
# _bucket_stats averages those rows at the seed level before bucketing.
BEST_PROMPT_GROUP_COLS = ["dataset", "classes_subset", "family", "family_config", "prompt_type"]
best_expl_stats = (
    best_expl_rows.groupby(BEST_PROMPT_GROUP_COLS, dropna=False)
    .apply(_bucket_stats, include_groups=False)
    .reset_index()
    if not best_expl_rows.empty
    else pd.DataFrame(columns=BEST_PROMPT_GROUP_COLS + ["mean_score", "std_score", "n_buckets", "n_seeds"])
)

# Non-anonymized baselines are not family- or specification-specific; include
# them once, averaging over equal keys/specifications at the seed level.
baseline_rows = df_f[(df_f["family_raw"] == "baseline") & df_f["prompt_type"].isin(NON_ANON_BASELINE_PROMPT_TYPES)].copy()
baseline_rows["family"] = "baseline"
baseline_rows["family_config"] = "baseline"
BASELINE_GROUP_COLS = ["dataset", "classes_subset", "family", "family_config", "prompt_type"]
baseline_stats = (
    baseline_rows.groupby(BASELINE_GROUP_COLS, dropna=False)
    .apply(_bucket_stats, include_groups=False)
    .reset_index()
    if not baseline_rows.empty
    else pd.DataFrame(columns=BASELINE_GROUP_COLS + ["mean_score", "std_score", "n_buckets", "n_seeds"])
)

best_plot_stats = pd.concat([best_expl_stats, baseline_stats], ignore_index=True)
best_plot_stats = best_plot_stats.dropna(subset=["mean_score"]).copy()
best_plot_stats

In [ ]:
def _short_config(config: str, max_len: int = 24) -> str:
    config = str(config)
    return config if len(config) <= max_len else config[: max_len - 1] + "…"

def _plot_best_family_panel(sub: pd.DataFrame, title: str) -> None:
    sub = sub.sort_values("mean_score", ascending=False).reset_index(drop=True)
    if sub.empty:
        print(f"skip empty panel: {title}")
        return

    labels = [
        f"{row.family}\n{_short_config(row.family_config)}\n{row.prompt_type}"
        for row in sub.itertuples(index=False)
    ]
    family_colors = {
        "concepts": "#4c72b0",
        "rationales": "#55a868",
        "attributions": "#c44e52",
        "baseline": "#333333",
    }
    colors = [family_colors.get(str(f), "#999999") for f in sub["family"]]
    x = np.arange(len(sub))

    fig, ax = plt.subplots(figsize=(max(8, 0.65 * len(sub) + 2), 5))
    ax.bar(
        x,
        sub["mean_score"],
        yerr=sub["std_score"],
        capsize=4,
        color=colors,
        edgecolor="black",
        linewidth=0.6,
    )
    ax.set_ylim(0, 1.05)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("score")
    ax.set_title(title + "\nranked highest to lowest; error bars = std over 10-seed buckets")
    ax.grid(axis="y", linestyle=":", alpha=0.5)

    present_families = [f for f in ["concepts", "rationales", "attributions", "baseline"] if f in set(sub["family"])]
    handles = [plt.Rectangle((0, 0), 1, 1, color=family_colors[f], ec="black") for f in present_families]
    ax.legend(handles, present_families, loc="upper right")
    fig.tight_layout()
    plt.show()

for (dataset, classes_subset), sub in best_plot_stats.groupby(["dataset", "classes_subset"]):
    _plot_best_family_panel(
        sub,
        title=f"Best family configs + baselines | {dataset} | classes_subset={classes_subset} | judge={MODEL_FILE}",
    )